<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/tapvidmv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TAPVid-MV Benchmark

Turns the pipeline's outputs (`depth/`, `extrinsics/`, `tracks/`) into the 50-episode released TAPVid-MV evaluation set.

```
metrics.json ──▶ 1. Shortlist ──▶ episodes_eval150.txt
                                       │
                                       ▼
                 2. Pick by eye ──▶ episodes_eval50.txt
                                       │
                                       ▼
                 3. Export ───────▶ data/output/tapvidmv/release/
```

Every output generated by this notebook lives under `data/output/tapvidmv/` (`episodes_eval150.txt`, `episodes_eval50.txt`, `previews/`, and `release/`), which is gitignored under `data/`.

## Prerequisites

Stages 1–3 of the pipeline (`compute_depth.py`, `compute_extrinsics.py`, `compute_tracks.py`) must already be written to `config.paths` locally or on GCS (`gs://dm-tapnet/tmp/droid_v2`).

---
## 0. Environment

Reads the mounted bucket on the pipeline machine, or pulls each episode's files from GCS on Colab.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
  subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "https://github.com/yangyi02/droid.git", "/content/droid"],
    check=True,
  )
  os.chdir("/content/droid")
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ml_collections", "mediapy"], check=True)
  from google.colab import auth, output
  auth.authenticate_user()
  output.enable_custom_widget_manager()

REPO = os.getcwd()
if REPO not in sys.path:
  sys.path.insert(0, REPO)

from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import functools
import glob
import json
import operator
import re
import shutil
import threading

import cv2
from IPython.display import display
import ipywidgets as widgets
import mediapy
import numpy as np
from tqdm.auto import tqdm

from config import get_config
import core.geometry
import core.io
import core.tracks
import core.visualization

config = get_config()

N_POOL = 150
N_TARGET = 50
MAX_PER_SCENE = 6
CLIP_WIDTH = 640
PREFETCH = 5
EXPORT_WORKERS = 8
JPEG_QUALITY = 95

OUT_DIR = os.path.join(REPO, "data", "output", "tapvidmv")
POOL_LIST = os.path.join(OUT_DIR, f"episodes_eval{N_POOL}.txt")
EVAL_LIST = os.path.join(OUT_DIR, f"episodes_eval{N_TARGET}.txt")
CACHE_DIR = os.path.join(OUT_DIR, "previews")
RELEASE_DIR = os.path.join(OUT_DIR, "release")
GCS_ROOT = "gs://dm-tapnet/tmp/droid_v2"

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"output: {OUT_DIR}")

---
## 1. Shortlist

Narrows all calibrated episodes down to `episodes_eval150.txt` in two steps:

| Cut | Threshold | Why |
|---|---|---|
| `robot_loss` | `<= 0.016` | Direct camera-robot alignment; robot tracks rest on it |
| `overlap` | `>= 43.0` | Cross-camera agreement; static points and visibility labels rest on it |
| `chamfer` | `<= 0.048` | Scene clutter plus alignment; kept loose so whole scenes are not dropped |

Survivors are grouped by **scene** (middle field of `episode_id`), ordered by weighted quality (`sum(value / limit)`), and drawn round-robin up to `MAX_PER_SCENE = 6`. If `episodes_eval150.txt` already exists under `data/output/tapvidmv/`, it is reused directly.

In [ ]:
OPS = {"<=": operator.le, ">=": operator.ge}
CUTS = {
  "chamfer": ("<=", 0.048),
  "overlap": (">=", 43.0),
  "robot_loss": ("<=", 0.016),
}


def cut_value(row, column, op):
  vals = [
    float(v)
    for k, v in row.items()
    if re.fullmatch(rf"{column}(_\d+)+", k) and v not in (None, "", "nan") and not np.isnan(float(v))
  ]
  if not vals:
    return float("nan")
  return min(vals) if op == ">=" else max(vals)


def quality(row, cuts):
  return sum(
    cut_value(row, col, op) / limit if op == "<=" else limit / max(cut_value(row, col, op), 1e-6)
    for col, (op, limit) in cuts.items()
  )


def sample_diverse(all_rows, n_target, cuts, max_per_scene):
  kept = [r for r in all_rows if all(OPS[op](cut_value(r, col, op), lim) for col, (op, lim) in cuts.items())]
  print(f"{len(kept)}/{len(all_rows)} episodes pass quality cuts")

  by_scene = {}
  for row in kept:
    by_scene.setdefault(row["scene"], []).append(row)
  for scene_rows in by_scene.values():
    scene_rows.sort(key=lambda r: quality(r, cuts))

  scenes = [s for _, s in sorted((-len(rs), s) for s, rs in by_scene.items())]
  selected, round_idx = [], 0
  while len(selected) < n_target and round_idx < max_per_scene:
    for scene in scenes:
      if round_idx < len(by_scene[scene]):
        selected.append(by_scene[scene][round_idx])
        if len(selected) == n_target:
          break
    round_idx += 1
  return selected


def get_extrinsics_root():
  local = config.paths.extrinsics
  if os.path.isdir(local):
    return local
  dst = "/content/episodes/extrinsics"
  os.makedirs(dst, exist_ok=True)
  subprocess.run(
    ["gcloud", "storage", "rsync", "-r", "--exclude", r"(?!.*metrics\.json$).+", f"{GCS_ROOT}/extrinsics", dst],
    check=True,
  )
  return dst


ext_root = get_extrinsics_root()
if os.path.exists(POOL_LIST):
  with open(POOL_LIST) as f:
    pool_ids = [line.strip() for line in f if line.strip()]
  rows = [json.load(open(os.path.join(ext_root, eid, "metrics.json"))) for eid in pool_ids]
  print(f"Loaded existing pool of {len(rows)} episodes from {POOL_LIST}")
else:
  tracks_dir = config.paths.tracks
  has_tracks = (
    {e for e in os.listdir(tracks_dir) if os.path.exists(os.path.join(tracks_dir, e, "tracks.npz"))}
    if os.path.isdir(tracks_dir)
    else None
  )
  all_rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(ext_root, "*", "metrics.json")))]
  if has_tracks is not None:
    all_rows = [r for r in all_rows if r["episode_id"] in has_tracks]
  rows = sample_diverse(all_rows, N_POOL, CUTS, MAX_PER_SCENE)
  with open(POOL_LIST, "w") as f:
    f.writelines(r["episode_id"] + "\n" for r in rows)
  print(f"Wrote {len(rows)} candidates over {len({r['scene'] for r in rows})} scenes to {POOL_LIST}")
  print(f"Sites: {dict(sorted(Counter(r['site'] for r in rows).items()))}")

---
## 2. Pick by eye

Quality cuts check the pipeline's self-consistency, so an episode whose tracks are confidently wrong or whose manipulation is uninteresting still passes. Each candidate plays as a three-view clip with ground-truth tracks drawn on it:
- **Blue / Green**: visible point on the robot arm / static background
- **Yellow / Red**: occluded point on the robot arm / static background

**Keep** takes the episode, **Skip** drops it, **Back** undoes the last call. Clips are prefetched in the background under `data/output/tapvidmv/previews/`.

In [ ]:
def _pull(episode_id, stage, exclude=None):
  local = os.path.join("/content/episodes", stage, episode_id)
  if exclude is not None and os.path.isdir(local):
    return local
  os.makedirs(local, exist_ok=True)
  cmd = ["gcloud", "storage", "rsync", "-r"]
  if exclude:
    cmd += ["--exclude", exclude]
  subprocess.run(cmd + [f"{GCS_ROOT}/{stage}/{episode_id}", local], check=True, capture_output=True)
  return local


def episode_roots(episode_id, full=False):
  mounted_depth = config.paths.depth
  if os.path.isdir(os.path.join(mounted_depth, episode_id)):
    return (
      mounted_depth,
      config.paths.extrinsics,
      config.paths.tracks,
    )
  depth_excl = None if full else r"(?!.*/(video_left\.mp4|calibration\.npz)$).+"
  tracks_excl = None if full else r".*robot_mask\.npz$"
  depth = _pull(episode_id, "depth", exclude=depth_excl)
  extrinsics = _pull(episode_id, "extrinsics")
  tracks = _pull(episode_id, "tracks", exclude=tracks_excl)
  return os.path.dirname(depth), os.path.dirname(extrinsics), os.path.dirname(tracks)


def read_video(path, width):
  capture = cv2.VideoCapture(path)
  frames, scale = [], 1.0
  while True:
    ok, frame = capture.read()
    if not ok:
      break
    scale = width / frame.shape[1]
    height = max(1, round(frame.shape[0] * scale))
    frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (width, height), interpolation=cv2.INTER_AREA))
  capture.release()
  return frames, scale


def draw_points(image, xy, colors, radius=5):
  canvas = image.copy()
  height, width = canvas.shape[:2]
  for (x, y), color in zip(np.round(xy).astype(int), colors):
    if -radius <= x < width + radius and -radius <= y < height + radius:
      cv2.circle(canvas, (int(x), int(y)), radius, tuple(int(c) for c in color), -1, cv2.LINE_AA)
  return canvas


def render_clip(episode_id):
  depth_root, ext_dir, tracks_root = episode_roots(episode_id, full=False)
  tracks = core.tracks.sample_tracks(
    core.io.load_track_data(episode_id, tracks_root), config.tracks.points_per_class
  )
  tracks_3d, vis = tracks["tracks_3d"], tracks["vis"]
  is_robot = np.arange(tracks_3d.shape[1]) < tracks["n_robot"]

  episode_dir = os.path.join(depth_root, episode_id)
  cams = sorted(c for c in os.listdir(episode_dir) if os.path.isdir(os.path.join(episode_dir, c)))

  uv, z = [], []
  for cam in cams:
    K = np.load(os.path.join(depth_root, episode_id, cam, "calibration.npz"))["K"].astype(np.float32)
    with open(os.path.join(ext_dir, episode_id, cam, "extrinsics.json")) as f:
      T_cam2world = np.array(json.load(f)["T_cam2world"], dtype=np.float32)
    cam_uv, cam_z = core.geometry.project_tracks(tracks_3d, K, T_cam2world)
    uv.append(cam_uv)
    z.append(cam_z)
  uv, z = np.stack(uv), np.stack(z)

  decoded, scales = zip(*(
    read_video(os.path.join(depth_root, episode_id, cam, "video_left.mp4"), CLIP_WIDTH) for cam in cams
  ))

  frames = []
  for t in range(uv.shape[1]):
    panels = []
    for view, cam in enumerate(cams):
      front = z[view, t] > 0
      panel = draw_points(
        decoded[view][t],
        uv[view, t, front] * scales[view],
        colors=core.visualization.occlusion_colors(is_robot[front], vis[view, t, front]),
      )
      core.visualization.draw_label(panel, f"view {view} [{cam}]", (6, 20), 0.5, (255, 255, 255), 1, 3)
      panels.append(panel)
    frames.append(np.hstack(panels))
  return np.stack(frames)


def ensure_clip(episode_id):
  path = os.path.join(CACHE_DIR, episode_id.replace("+", "_") + ".mp4")
  if os.path.exists(path):
    return path
  try:
    mediapy.write_video(path, render_clip(episode_id), fps=12)
  except Exception as failure:
    print(f"  {episode_id}: {type(failure).__name__}: {failure}")
    return None
  return path


pool = ThreadPoolExecutor(max_workers=3)
_futures = {}
_lock = threading.Lock()


def prefetch(index):
  with _lock:
    for row in rows[index : index + PREFETCH]:
      episode_id = row["episode_id"]
      if episode_id not in _futures:
        _futures[episode_id] = pool.submit(ensure_clip, episode_id)


def clip_bytes(episode_id):
  with _lock:
    future = _futures.get(episode_id)
  path = future.result() if future is not None else ensure_clip(episode_id)
  return open(path, "rb").read() if path else None


prefetch(0)
print(f"Prefetching first {PREFETCH} clips into {CACHE_DIR}")

In [ ]:
def _f(row, key, fmt="{:.3f}"):
  try:
    return fmt.format(float(row[key]))
  except (KeyError, TypeError, ValueError):
    return "-"


def _worst(row, column, fmt="{:.1f}"):
  value = cut_value(row, column, CUTS[column][0])
  return "-" if not np.isfinite(value) else fmt.format(value)


@functools.lru_cache(maxsize=None)
def track_stats(episode_id):
  _, _, tracks_root = episode_roots(episode_id, full=False)
  tracks = core.tracks.sample_tracks(
    core.io.load_track_data(episode_id, tracks_root), config.tracks.points_per_class
  )
  vis = tracks["vis"]
  seen_by_someone = vis.any(axis=0).mean(axis=0)
  return {
    "points": vis.shape[2],
    "robot": tracks["n_robot"] / vis.shape[2],
    "visible": float(vis.mean()),
    "unseen": float((seen_by_someone < 0.5).mean()),
  }


def summarise(row):
  episode_id = row["episode_id"]
  scene = episode_id.split("+")[1]
  stats = track_stats(episode_id)
  taken = sum(kept.split("+")[1] == scene for kept in state["kept"])
  pool_here = sum(r["episode_id"].split("+")[1] == scene for r in rows)
  return (
    f"<b>{episode_id}</b><br>"
    f"scene {scene} &middot; site {row.get('site', '?')} &middot; "
    f"<b>{taken} kept from this scene</b> of {pool_here} in the pool<br>"
    f"{_f(row, 'n_frames', '{:.0f}')} frames &middot; {stats['points']} points &middot; "
    f"{100 * stats['robot']:.0f}% on the arm &middot; {100 * stats['visible']:.0f}% annotated visible &middot; "
    f"{100 * stats['unseen']:.1f}% of points no camera sees for half the episode<br>"
    f"chamfer {_worst(row, 'chamfer', '{:.3f}')} &middot; overlap {_worst(row, 'overlap', '{:.1f}')} &middot; "
    f"robot loss {_worst(row, 'robot_loss', '{:.4f}')}"
  )


state = {"index": 0, "kept": [], "history": []}

video = widgets.Video(format="mp4", autoplay=True, loop=True, width=3 * CLIP_WIDTH)
info = widgets.HTML()
progress = widgets.HTML()
keep_button = widgets.Button(description="Keep", button_style="success", icon="check")
skip_button = widgets.Button(description="Skip", button_style="danger", icon="times")
back_button = widgets.Button(description="Back", icon="undo")


def show():
  index = state["index"]
  scenes_kept = len({kept.split("+")[1] for kept in state["kept"]})
  progress.value = (
    f"<b>{len(state['kept'])}</b> / {N_TARGET} kept over <b>{scenes_kept}</b> scenes "
    f"(the pool has {len({r['episode_id'].split('+')[1] for r in rows})}) &nbsp;&middot;&nbsp; "
    f"candidate <b>{min(index + 1, len(rows))}</b> / {len(rows)}"
  )
  if index >= len(rows) or len(state["kept"]) >= N_TARGET:
    info.value = "<b>Done.</b> Run the next cell to write the list."
    video.layout.display = "none"
    for button in (keep_button, skip_button, back_button):
      button.disabled = button is not back_button
    return

  row = rows[index]
  info.value = summarise(row)
  prefetch(index)
  payload = clip_bytes(row["episode_id"])
  if payload is None:
    info.value += "<br><i>clip failed to render - skipping</i>"
    state["index"] += 1
    show()
    return
  video.layout.display = ""
  video.value = payload


def decide(keep):
  index = state["index"]
  if index >= len(rows):
    return
  state["history"].append((index, keep))
  if keep:
    state["kept"].append(rows[index]["episode_id"])
  state["index"] = index + 1
  show()


def undo(_):
  if not state["history"]:
    return
  index, keep = state["history"].pop()
  if keep and state["kept"]:
    state["kept"].pop()
  state["index"] = index
  for button in (keep_button, skip_button):
    button.disabled = False
  show()


keep_button.on_click(lambda _: decide(True))
skip_button.on_click(lambda _: decide(False))
back_button.on_click(undo)

display(
  widgets.VBox([
    progress,
    video,
    info,
    widgets.HBox([keep_button, skip_button, back_button]),
  ])
)
show()

In [ ]:
kept = state["kept"]
if not kept:
  if os.path.exists(EVAL_LIST):
    with open(EVAL_LIST) as f:
      kept = [line.strip() for line in f if line.strip()]
    print(f"Kept existing {len(kept)} episodes in {EVAL_LIST}")
  else:
    print("Nothing kept - not writing.")
else:
  scenes = {eid.split("+")[1] for eid in kept}
  sites = {eid.split("+")[0] for eid in kept}
  with open(EVAL_LIST, "w") as f:
    f.writelines(eid + "\n" for eid in sorted(kept))
  print(f"Wrote {len(kept)} episodes over {len(scenes)} scenes and {len(sites)} sites to {EVAL_LIST}")

---
## 3. Export

Writes the selected 50 episodes into `data/output/tapvidmv/release/` across `EXPORT_WORKERS = 8` threads (~2 min total) and lists every exported `.npy` / `.npz` file in `droid_file_list.txt`. Each episode writes to a `.partial` staging directory and renames atomically when complete, so re-running resumes cleanly.

```
<episode_id>/
├── tracks_xyz.npy          (frames, points, 3)   world-space 3D tracks
├── queries_xytv.npy        (points, 4)           x, y, t, query view
└── <view>/                 one directory per camera, 0-indexed
    ├── images_jpeg_bytes.npy   (frames,)  object array of JPEG buffers
    ├── intrinsics.npy          (4,)       fx, fy, cx, cy
    ├── extrinsics_w2c.npy      (frames, 4, 4)
    ├── visibility.npy          (frames, points)
    ├── depth.npz               "depth": (frames, height, width) uint16 mm, 0 = no reading
    └── foreground_mask.npz     "mask":  (frames, height, width) bool, rendered robot arm
```

Publishing to `gs://dm-tapnet/tapvidmv/droid/` and `gs://dm-tapnet/tapvidmv/droid_file_list.txt` is done by hand after export.

In [ ]:
def _encode_jpeg(rgb_frame, quality=JPEG_QUALITY):
  bgr = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR)
  _, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
  return np.frombuffer(buf, dtype=np.uint8).copy()


def export_episode(episode_id):
  seq_dir = os.path.join(RELEASE_DIR, episode_id)
  if os.path.isdir(seq_dir):
    return seq_dir

  depth_root, ext_dir, tracks_root = episode_roots(episode_id, full=True)
  episode = core.io.load_depth_data(episode_id, depth_root, load_video=True)
  poses = core.io.load_extrinsics(episode, ext_dir)
  tracks = core.tracks.sample_tracks(
    core.io.load_track_data(episode_id, tracks_root), config.tracks.points_per_class
  )
  uv = core.tracks.project_views(tracks["tracks_3d"], episode, poses)
  masks = core.io.load_robot_masks(episode_id, tracks_root, episode["camera"])

  staging = seq_dir + ".partial"
  shutil.rmtree(staging, ignore_errors=True)
  os.makedirs(staging)

  tracks_3d = tracks["tracks_3d"].astype(np.float32)
  np.save(os.path.join(staging, "tracks_xyz.npy"), tracks_3d)
  queries = core.tracks.build_queries(uv, tracks["query_view"], tracks["query_frame"])
  np.save(os.path.join(staging, "queries_xytv.npy"), queries)

  for view, (cam_id, cam_data) in enumerate(episode["camera"].items()):
    view_dir = os.path.join(staging, str(view))
    os.makedirs(view_dir, exist_ok=True)

    video = cam_data["video_rgb"]
    jpeg_arr = np.empty(len(video), dtype=object)
    jpeg_arr[:] = [_encode_jpeg(frame) for frame in video]
    np.save(os.path.join(view_dir, "images_jpeg_bytes.npy"), jpeg_arr)

    K = cam_data["K"]
    intrinsics = np.array([K[0, 0], K[1, 1], K[0, 2], K[1, 2]], dtype=np.float32)
    np.save(os.path.join(view_dir, "intrinsics.npy"), intrinsics)

    T_world2cam = np.linalg.inv(poses[cam_id]["T_cam2world"]).astype(np.float32)
    np.save(os.path.join(view_dir, "extrinsics_w2c.npy"), T_world2cam)
    np.save(os.path.join(view_dir, "visibility.npy"), tracks["vis"][view].astype(bool))

    depth_mm = np.round(cam_data["raw_depth"] * 1000).astype(np.uint16)
    np.savez_compressed(os.path.join(view_dir, "depth.npz"), depth=depth_mm)
    np.savez_compressed(os.path.join(view_dir, "foreground_mask.npz"), mask=masks[cam_id])

  shutil.rmtree(seq_dir, ignore_errors=True)
  os.rename(staging, seq_dir)
  return seq_dir


os.makedirs(RELEASE_DIR, exist_ok=True)
with open(EVAL_LIST) as f:
  eval_ids = [line.strip() for line in f if line.strip()]

with ThreadPoolExecutor(max_workers=EXPORT_WORKERS) as executor:
  list(tqdm(executor.map(export_episode, eval_ids), total=len(eval_ids), desc="Exporting"))

rel_files = []
for root, dirs, files in os.walk(RELEASE_DIR):
  dirs[:] = [d for d in dirs if not d.endswith(".partial")]
  for name in files:
    if name.endswith((".npy", ".npz")):
      rel_files.append(os.path.relpath(os.path.join(root, name), RELEASE_DIR))

file_list_path = os.path.join(RELEASE_DIR, "droid_file_list.txt")
with open(file_list_path, "w") as f:
  f.writelines(p + "\n" for p in sorted(rel_files))

print(f"Exported {len(eval_ids)} episodes ({len(rel_files)} files) to {RELEASE_DIR}")
print(f"Wrote {file_list_path}")